# Retirement Readiness
# 1. Data Preprocessing

Notebook 1 showed us what is in the data and which problems need to be handled before modeling. This notebook turns those findings into a **repeatable preprocessing pipeline**.

The raw records cannot go directly into a model because they contain missing values, categorical text, identifiers, target-derived columns, and numeric variables on very different scales.

The main rule throughout this notebook is simple:

> **If a preprocessing step has to learn something from the data, it learns it from the training customers only.**

For example, the median used to fill a missing value is learned from training data. The test set only receives that already-learned rule. This prevents information from the test set from leaking into the model.

### Decisions carried forward from Notebook 1

| Finding | Action |
|---|---|
| `Funding_Gap`, `Readiness_Score` and `RetirementReady` are calculated from the target | Remove them from the feature set |
| `CustomerID` identifies a customer but does not describe their financial situation | Remove it |
| 150 records are exact duplicates | Remove them before the split |
| Twelve numeric columns have roughly 1.5% missing values | Use median imputation learned from training data |
| The target is strongly right-skewed | Keep the raw target and create its natural log for modeling |
| `YearsUntilRetirement` is not provided directly | Engineer it from `DesiredRetirementAge - Age` |
| `Age` and `YearsExperience` correlate at +0.94 | Engineer `CareerStartAge` |
| The education relationship is not monotonic | One-hot encode categorical variables |

By the end of the notebook, we will have a clean numeric feature matrix and saved preprocessing pipelines that Notebook 3 can use for model training.


---

# 2. Load Data

### What are we doing?

We load the project dataset and locate the reusable feature-engineering code.

The project is organized like this:

```text
Retirement-Readiness-Predictor/
├── data/retirement_dataset_v2.csv
├── notebooks/02_data_preprocessing.ipynb
├── src/feature_engineering.py
├── artifacts/
└── figures/
```

The notebook searches for the repository root instead of depending on one fixed working directory. This allows it to run from either the repository root or the `notebooks/` folder.

In Google Colab, the repository can be cloned first, and the same path logic will locate it.

The important point is that **the data and reusable feature-engineering code live outside the notebook**, so the same logic can later be reused when the saved pipeline is loaded.


In [ ]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn import set_config
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
import joblib

# Display fitted pipelines as a diagram rather than a block of text.
set_config(display="diagram")

print("pandas       ", pd.__version__)
print("numpy        ", np.__version__)
print("scikit-learn ", sklearn.__version__)

In [ ]:
# The repository root is the folder that contains src/. Checking a short list
# of candidates keeps the notebook working from either the root or notebooks/.
CANDIDATE_ROOTS = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/Retirement-Readiness-Predictor"),   # Colab, after cloning
]

PROJECT_ROOT = None
for candidate in CANDIDATE_ROOTS:
    if (candidate / "src" / "feature_engineering.py").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root (the folder containing src/). Checked: "
        + ", ".join(str(path) for path in CANDIDATE_ROOTS)
    )

# Make src/ importable no matter where the notebook was launched from.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
FIGURES_DIR = PROJECT_ROOT / "figures"
ARTIFACTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
# The feature-engineering code lives in src/ so that the saved pipeline can be
# reloaded later without needing this notebook.
from src.feature_engineering import (
    ENGINEERED_FEATURES,
    SOURCE_COLUMNS_TO_DROP,
    add_engineered_features,
    engineered_feature_names,
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300})

NAVY = "#1F3B73"
TEAL = "#2A9D8F"
AMBER = "#E9A13B"
CORAL = "#E76F51"
SLATE = "#6C757D"

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "retirement_dataset_v2.csv"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "retirement_dataset_v2.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"retirement_dataset_v2.csv not found under {PROJECT_ROOT}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

---

# 3. Separate Features and Target

### What are we doing?

We separate the column we want to predict from the columns used to make the prediction.

The target is:

`Expected_Retirement_Fund`

It represents the customer's projected retirement fund.

Keeping the target in a separate object prevents preprocessing steps intended for predictors from accidentally being applied to it.

### Why use the log of the target?

The EDA showed that the retirement-fund values are strongly right-skewed. Most customers have lower projected funds, while a smaller group reaches into the tens of millions.

The natural logarithm compresses this long right tail. Notebook 3 will train the baseline models using the log target, while the original dollar target is kept so predictions can later be converted back into financial terms.

Nothing is being predicted yet. We are only preparing the target and predictors for the modeling stage.


In [ ]:
TARGET = "Expected_Retirement_Fund"

# A missing or non-positive target could not be logged or used for training.
print(f"Missing target values:      {df[TARGET].isna().sum()}")
print(f"Non-positive target values: {(df[TARGET] <= 0).sum()}")
print(f"Skewness in dollars:        {df[TARGET].skew():+.4f}")
print(f"Skewness after log:         {np.log(df[TARGET]).skew():+.4f}")

---

# 4. Remove IDs and Leakage

### What are we doing?

We remove four columns from the predictors:

- `CustomerID`
- `Funding_Gap`
- `Readiness_Score`
- `RetirementReady`

### Why remove them?

`CustomerID` identifies a customer, but it does not describe their financial situation. Keeping an identifier can also allow a flexible model to memorize patterns that have no useful meaning.

The other three columns are more serious: they are calculated using the retirement-fund outcome.

- `Funding_Gap` = projected fund − retirement goal
- `Readiness_Score` = projected fund ÷ retirement goal
- `RetirementReady` = whether the funding gap is positive

If we gave these columns to the model, we would be giving it information derived from the answer it is supposed to predict. This is **target leakage**.

The cells below also reconfirm the arithmetic relationships so the decision is based on the actual dataset.

### Decision

These four columns leave the feature set. The business measures themselves are not discarded from the project; they can be calculated **after** a model produces a prediction.

### Why keep `Retirement_Fund_Goal`?

The project assumes that a customer's retirement goal is known before a projection is produced, for example through an adviser.

Under that business-process assumption, it is a valid input.

Notebook 3 will run a small ablation experiment without this variable to measure how much the model actually depends on it.


In [ ]:
IDENTIFIER_COLUMNS = ["CustomerID"]
KPI_COLUMNS = ["Readiness_Score", "Funding_Gap", "RetirementReady"]

# Kept as an input, recorded here so Notebook 3 can run the ablation.
ABLATION_REVIEW_COLUMNS = ["Retirement_Fund_Goal"]

gap_error = (df[TARGET] - df.Retirement_Fund_Goal - df.Funding_Gap).abs().max()
ready_match = ((df.Funding_Gap >= 0).astype(int) == df.RetirementReady).mean()

print(f"max |Funding_Gap - (target - goal)|:    {gap_error:.10f}")
print(f"RetirementReady == 1[Funding_Gap >= 0]: {ready_match:.4%} of rows")

In [ ]:
columns_to_exclude = IDENTIFIER_COLUMNS + KPI_COLUMNS + [TARGET]

X = df.drop(columns=columns_to_exclude)
y = df[TARGET]

# Column roles come from the dtypes, so a change to the source file cannot
# leave a column silently unprocessed.
CATEGORICAL_FEATURES = X.select_dtypes(include="object").columns.tolist()
NUMERIC_FEATURES = X.select_dtypes(include=np.number).columns.tolist()

print(f"Predictor columns: {X.shape[1]}  "
      f"({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)")
print("Excluded:", ", ".join(columns_to_exclude))

---

# 5. Remove Duplicates

### What are we doing?

The dataset contains **150 exact duplicate records**.

We remove them before the train/test split.

### Why?

A duplicate record gives the same observation extra weight during training. More importantly, if duplicates are split across training and test data, essentially the same customer record could appear on both sides.

That would make the test result look better than performance on genuinely unseen customers.

The duplicates are identified using the complete original record. The same rows are removed from both `X` and `y` so that the predictors and target remain perfectly aligned.


In [ ]:
duplicate_mask = df.duplicated(keep="first")
rows_to_keep = ~duplicate_mask

print(f"Exact duplicate records: {duplicate_mask.sum()}")
print(f"Rows before: {len(X):,}")

X = X[rows_to_keep]
y = y[rows_to_keep]

print(f"Rows after:  {len(X):,}")
print(f"Duplicates remaining in X: {X.duplicated().sum()}")
print(f"X and y aligned: {X.index.equals(y.index)}")

---

# 6. Train/Test Split

### What are we doing?

We set aside **20% of the customers as the test set**.

These records will not be used to learn preprocessing rules or choose the baseline model. They are kept until the final evaluation in Notebook 3.

### Why split before preprocessing?

Imputation, scaling, and category encoding all involve information learned from the data.

If we preprocess the entire dataset first, the test customers could influence those learned values. That would be a form of data leakage.

So the order is:

```text
Raw data
   ↓
Train / test split
   ↓
Learn preprocessing rules from training data only
   ↓
Apply those rules to train and test
```

`random_state=42` makes the split reproducible.

The target is a continuous amount, not a class label, so there is no class to stratify. Instead, the next check compares the train and test samples on important variables to make sure the random split is reasonably representative.


In [ ]:
X_train, X_test, y_train_dollars, y_test_dollars = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

# The target must be present and strictly positive because we model its natural logarithm.
assert y_train_dollars.notna().all(), "Training target contains missing values."
assert y_test_dollars.notna().all(), "Test target contains missing values."
assert (y_train_dollars > 0).all(), "Training target must be strictly positive."
assert (y_test_dollars > 0).all(), "Test target must be strictly positive."

# Model training uses the log target; dollars are kept for later reporting.
y_train = np.log(y_train_dollars)
y_test = np.log(y_test_dollars)

print(f"Training set: {X_train.shape[0]:,} rows ({X_train.shape[0] / len(X):.0%})")
print(f"Test set:     {X_test.shape[0]:,} rows ({X_test.shape[0] / len(X):.0%})")
print(f"y_train (log): mean {y_train.mean():.4f}, std {y_train.std():.4f}")
print(f"y_test  (log): mean {y_test.mean():.4f}, std {y_test.std():.4f}")

In [ ]:
# A quick balance check: compare the middle value of several features across
# the two samples. Large differences would suggest an unlucky split.
comparison_features = [
    "AnnualSalary", "Savings", "RetirementAccountBalance", "EmergencyFund",
    "Age", "SavingsRate", "CreditScore", "MonthlyExpenses",
]

comparison_rows = []
for feature in comparison_features:
    train_median = X_train[feature].median()
    test_median = X_test[feature].median()
    comparison_rows.append({
        "feature": feature,
        "train_median": train_median,
        "test_median": test_median,
        "pct_difference": (test_median / train_median - 1) * 100,
    })

split_comparison = pd.DataFrame(comparison_rows).set_index("feature")
split_comparison.round(3)

### Result

The selected variables show no large differences between the training and test samples.

That supports using this random split as the project's train/test division.

This check does not prove that the two samples are identical. It simply helps us confirm that the split did not produce an obviously unusual test group.


---

# 7. Handle Missing Values

### What are we doing?

Some customers have missing values in financial and categorical columns.

Instead of deleting incomplete customers, we fill the missing values:

- **Numeric columns:** use the training median.
- **Categorical columns:** use the most common training category.

### Why?

Removing every incomplete record would discard roughly **one customer in six** even though only about **1.5% of individual numeric values** are missing.

The EDA also found a missingness pattern consistent with **MCAR**. For this baseline, simple imputation is therefore a reasonable choice.

The median is especially useful for financial variables because several of them are strongly skewed. A small number of very wealthy customers can pull the mean upward, while the median remains much more representative of the middle of the distribution.

### Decision

We use:

```python
SimpleImputer(strategy="median")
```

for numeric features and:

```python
SimpleImputer(strategy="most_frequent")
```

for categorical features.

The important part is **when these imputers are fitted**. They are created here, but they are fitted later inside the preprocessing pipeline using the training data only.


In [ ]:
train_missing = X_train.isna().sum()
affected_columns = train_missing[train_missing > 0].index.tolist()

missing_rows = []
for column in affected_columns:
    missing_rows.append({
        "column": column,
        "n_missing": int(X_train[column].isna().sum()),
        "pct_missing": X_train[column].isna().mean() * 100,
        "train_median": X_train[column].median(),
        "train_mean": X_train[column].mean(),
    })

missing_summary = pd.DataFrame(missing_rows).set_index("column")

print(f"Numeric columns with gaps:     {len(affected_columns)}")
print(f"Missing values in training set: {X_train.isna().sum().sum():,} "
      f"({X_train.isna().sum().sum() / X_train.size * 100:.2f}% of all values)")
print(f"Training rows with any gap:     {X_train.isna().any(axis=1).sum():,} "
      f"({X_train.isna().any(axis=1).mean() * 100:.1f}%)")
print()
missing_summary.round(2)

In [ ]:
numeric_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

---

# 8. Feature Engineering

### What are we doing?

We create **five new features** from information that already exists in each customer's record.

Feature engineering means giving the model a useful relationship explicitly instead of expecting every algorithm to discover it on its own.

For example, `AnnualSalary` and `SavingsRate` are two separate columns. Their product has a direct financial interpretation, so we create that product as a new feature.

### The five engineered features

| Feature | Calculation | What it represents |
|---|---|---|
| `YearsUntilRetirement` | `DesiredRetirementAge − Age` | How many years the customer's money still has to grow. |
| `CareerStartAge` | `Age − YearsExperience` | Approximate age when the customer started working. |
| `SalaryBasedContribution` | `AnnualSalary × SavingsRate` | Salary-based savings implied by the savings rate; this is a proxy, not a recorded contribution. |
| `RealExpectedReturn` | `ExpectedAnnualReturn − ExpectedInflation` | Expected investment growth after inflation. |
| `DebtToIncomeRatio` | `MortgageBalance ÷ AnnualSalary` | Mortgage size relative to earnings. |

These features are deliberately limited to relationships that have a clear financial interpretation.

### Why remove three original columns afterward?

Once the engineered features are created, `DesiredRetirementAge`, `YearsExperience`, and `ExpectedInflation` are removed.

Each of these is an exact component of one of the engineered features. Keeping both versions would duplicate information and can make regularized linear coefficients harder to interpret.

This is **not** a rule that "correlated features must always be removed." We only remove these columns because their information has been explicitly represented in a new feature.

### When does feature engineering happen?

It happens **after the first imputation step**.

That order matters because a missing input could otherwise produce a missing engineered feature. The pipeline later applies a second imputer to the engineered values.

The calculations use only values from the same customer's row, so feature engineering itself does not learn anything from the dataset.

### What about customers with no salary?

There are **835 customers with `AnnualSalary == 0`**. They form the unemployed group rather than scattered damaged records.

For these customers, `MortgageBalance ÷ AnnualSalary` would involve division by zero. Instead of creating infinity, the denominator is treated as missing, so `DebtToIncomeRatio` becomes missing and is handled by the second median imputer.

The categorical variable `EmploymentStatus_Unemployed` is retained after encoding, so the model still has information identifying this group.


In [ ]:
feature_engineering_step = FunctionTransformer(
    add_engineered_features,
    feature_names_out=engineered_feature_names,
)

# Fills DebtToIncomeRatio for the customers with no salary.
engineered_imputer = SimpleImputer(strategy="median")

print("Engineered:", ", ".join(ENGINEERED_FEATURES))
print("Removed as exact components:", ", ".join(SOURCE_COLUMNS_TO_DROP))

In [ ]:
# Preview on the raw training columns. Inside the pipeline, engineering runs after
# numeric imputation, so the only new missing values it can introduce are undefined
# debt-to-income ratios for customers with zero salary.
engineered_preview = add_engineered_features(X_train[NUMERIC_FEATURES])

preview = engineered_preview[ENGINEERED_FEATURES].describe().T[["min", "50%", "max"]]

correlations = []
for feature in ENGINEERED_FEATURES:
    correlations.append(engineered_preview[feature].corr(y_train))
preview["corr_with_log_target"] = correlations

zero_salary_rows = X_train.AnnualSalary == 0
print(f"Training customers with no salary: {zero_salary_rows.sum():,} "
      f"({zero_salary_rows.mean() * 100:.2f}%)")
print("Their employment status:",
      X_train.loc[zero_salary_rows, "EmploymentStatus"].unique())
print("Infinite values produced:",
      int(np.isinf(engineered_preview[ENGINEERED_FEATURES]).sum().sum()))
print()
preview.round(4)

---

# 9. Encode Categorical Variables

### What are we doing?

Machine-learning models need numerical inputs, but columns such as `Education` and `EmploymentStatus` contain categories.

We convert each category into a **binary indicator column** containing 1 or 0.

For example, if `Education` has four levels, the encoder creates three indicator columns when `drop="first"` is used.

A customer with a Master's degree might therefore have:

```text
Education_HighSchool   0
Education_Bachelor     0
Education_Master       1
```

while a PhD customer could have:

```text
Education_HighSchool   0
Education_Bachelor     0
Education_Master       0
```

The dropped category is represented when all the remaining indicators are 0.

### Why drop one category?

If we created an indicator for every category, the columns would contain perfect redundancy: knowing all but one category already tells us the missing category.

Dropping one level avoids that unnecessary redundancy.

### What about a category we did not see during training?

`handle_unknown="ignore"` prevents an unseen category from breaking the pipeline. Its indicator columns are simply all zero.

This makes the preprocessing safer when the pipeline receives new customer records later.


In [ ]:
encoding_rows = []
for column in CATEGORICAL_FEATURES:
    levels = sorted(X_train[column].dropna().unique())
    encoding_rows.append({
        "column": column,
        "n_levels": len(levels),
        # Categories are sorted, so drop="first" removes this one.
        "dropped_level": levels[0],
        "columns_created": len(levels) - 1,
    })

encoding_plan = pd.DataFrame(encoding_rows).set_index("column")
print(f"Indicator columns to be created: {encoding_plan.columns_created.sum()}")
print()
encoding_plan

In [ ]:
categorical_encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False,
)

---

# 10. Scale Numerical Features

### What are we doing?

The numeric variables use very different units.

For example:

- `SavingsRate` is between 0 and 1.
- Retirement balances can be measured in millions of dollars.

Standardisation puts numeric features onto a common scale by expressing each value in terms of its distance from the feature's training mean, measured in training standard deviations.

In simple terms, a value of `+2` after scaling means:

> this value is about two standard deviations above the training average for that feature.

### Why?

Scaling is especially useful for models whose calculations depend on feature magnitude.

It matters for:

- regularized linear models, because their penalties depend on coefficient size
- distance-based models such as SVM and KNN

Ordinary Linear Regression does not mathematically require scaling, but using the same standardized representation keeps the linear pipeline consistent.

Tree models are different. A tree asks questions such as "Is this value greater than a threshold?" and does not depend on the relative scale of different features.

### Decision

We therefore keep **two preprocessing variants**:

- **Scaled pipeline:** for linear and distance-based models.
- **Unscaled pipeline:** for tree-based models.

Both variants use the same imputation, feature engineering, and categorical encoding.


In [ ]:
scale_overview = pd.DataFrame({
    "min": X_train[NUMERIC_FEATURES].min(),
    "median": X_train[NUMERIC_FEATURES].median(),
    "max": X_train[NUMERIC_FEATURES].max(),
    "std": X_train[NUMERIC_FEATURES].std(),
}).sort_values("std")

print(f"Smallest standard deviation: {scale_overview['std'].min():,.4f} "
      f"({scale_overview.index[0]})")
print(f"Largest standard deviation:  {scale_overview['std'].max():,.0f} "
      f"({scale_overview.index[-1]})")
print()
scale_overview.round(4)

In [ ]:
feature_scaler = StandardScaler()

---

# 11. Build the Preprocessing Pipeline

### What are we doing?

The preprocessing steps are now assembled into one object.

Numeric and categorical features follow separate paths:

```text
Numeric features
    ↓
Impute missing values
    ↓
Create engineered features
    ↓
Impute any new missing values
    ↓
Scale (or leave unscaled)
```

```text
Categorical features
    ↓
Fill missing categories
    ↓
One-hot encode
```

The two branches are then combined into one final feature matrix.

### Why use a pipeline?

A pipeline makes the order of operations explicit and repeatable.

More importantly, the same sequence is used during:

- training
- validation
- testing
- cross-validation
- future predictions

When cross-validation refits the pipeline, each fold learns its own preprocessing rules from its fitting data. That is how we avoid accidentally letting validation information influence preprocessing.

`remainder="drop"` is explicit here: any column that is not assigned to one of the two branches is discarded rather than silently passed through.


In [ ]:
# Numeric route: fill gaps, build features, fill the undefined ratio, scale.
numeric_pipeline = Pipeline(steps=[
    ("impute", numeric_imputer),
    ("engineer", feature_engineering_step),
    ("impute_engineered", engineered_imputer),
    ("scale", feature_scaler),
])

# Categorical route: fill gaps, convert to indicator columns.
categorical_pipeline = Pipeline(steps=[
    ("impute", categorical_imputer),
    ("encode", categorical_encoder),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,   # keep "Age", not "numeric__Age"
)

# Return DataFrames so column names survive the transformation.
preprocessor.set_output(transform="pandas")

preprocessor

In [ ]:
# The same pipeline with the scaling step switched off, for tree-based models.
preprocessor_unscaled = clone(preprocessor)
preprocessor_unscaled.set_params(numeric__scale="passthrough")
preprocessor_unscaled.set_output(transform="pandas")

print("scaled variant   ->", preprocessor.get_params()["numeric__scale"])
print("unscaled variant ->", preprocessor_unscaled.get_params()["numeric__scale"])

# How wide the output should be, derived from the configuration above. The
# verification section compares this with the matrix that is actually produced.
n_numeric_kept = len(NUMERIC_FEATURES) - len(SOURCE_COLUMNS_TO_DROP)
n_engineered = len(ENGINEERED_FEATURES)
n_numeric_total = n_numeric_kept + n_engineered
n_dummies = int(encoding_plan.columns_created.sum())
n_expected_features = n_numeric_total + n_dummies

print(f"\nExpected: {n_numeric_total} numeric + {n_dummies} indicator "
      f"= {n_expected_features} features")

In [ ]:
# --- Figure: the two routes through the pipeline --------------------------
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 100)
ax.set_ylim(0, 52)
ax.axis("off")


def draw_box(x, y, width, height, label, colour):
    ax.add_patch(plt.Rectangle((x, y), width, height, facecolor=colour,
                               edgecolor="white", linewidth=1.6, zorder=2))
    ax.text(x + width / 2, y + height / 2, label, ha="center", va="center",
            fontsize=9.5, color="white", fontweight="bold", zorder=3)


def draw_arrow(x_start, y, x_end):
    ax.annotate("", xy=(x_end, y), xytext=(x_start, y),
                arrowprops={"arrowstyle": "->", "color": SLATE, "linewidth": 1.6})


draw_box(1, 21, 13, 8, f"X_train\n{X_train.shape[0]:,} x {X_train.shape[1]}", NAVY)
draw_arrow(14.4, 25, 16.2)

ax.text(18, 45.5, f"NUMERIC  ({len(NUMERIC_FEATURES)} columns)", fontsize=10,
        color=NAVY, fontweight="bold")
numeric_stages = ["Fill gaps\n(median)", "Build 5 features\nremove 3 sources",
                  "Fill undefined\nratio", "Standardise"]
for index, stage in enumerate(numeric_stages):
    x = 18 + index * 16
    draw_box(x, 34, 13.5, 9, stage, TEAL)
    if index < len(numeric_stages) - 1:
        draw_arrow(x + 13.9, 38.5, x + 15.6)

ax.text(18, 15.5, f"CATEGORICAL  ({len(CATEGORICAL_FEATURES)} columns)",
        fontsize=10, color=NAVY, fontweight="bold")
categorical_stages = ["Fill gaps\n(most frequent)", "Indicator columns\ndrop first"]
for index, stage in enumerate(categorical_stages):
    x = 18 + index * 16
    draw_box(x, 4, 13.5, 9, stage, AMBER)
    if index < len(categorical_stages) - 1:
        draw_arrow(x + 13.9, 8.5, x + 15.6)

# Connectors splitting the input into two routes and merging them again.
ax.plot([16.2, 16.2], [8.5, 38.5], color=SLATE, lw=1.6, zorder=1)
ax.plot([16.2, 18], [38.5, 38.5], color=SLATE, lw=1.6, zorder=1)
ax.plot([16.2, 18], [8.5, 8.5], color=SLATE, lw=1.6, zorder=1)
ax.plot([79.5, 84], [38.5, 38.5], color=SLATE, lw=1.6, zorder=1)
ax.plot([47.5, 84], [8.5, 8.5], color=SLATE, lw=1.6, zorder=1)
ax.plot([84, 84], [8.5, 38.5], color=SLATE, lw=1.6, zorder=1)
draw_arrow(84, 25, 86)

draw_box(86, 21, 13, 8,
         f"processed\n{X_train.shape[0]:,} x {n_expected_features}", NAVY)

for x, label in [(24.8, len(NUMERIC_FEATURES)), (40.8, n_numeric_total),
                 (56.8, n_numeric_total), (72.8, n_numeric_total)]:
    ax.text(x, 31.5, f"{label} cols", ha="center", fontsize=8.5, color=SLATE)
for x, label in [(24.8, len(CATEGORICAL_FEATURES)), (40.8, n_dummies)]:
    ax.text(x, 1.2, f"{label} cols", ha="center", fontsize=8.5, color=SLATE)

ax.text(50, 21.5, "fitted on X_train    |    applied to X_train and X_test",
        ha="center", fontsize=11, color=CORAL, fontweight="bold")
ax.text(50, 18.2, "medians, scaling statistics and category lists all come "
        "from the training data",
        ha="center", fontsize=9, color=SLATE, style="italic")

ax.set_title("Preprocessing pipeline", fontsize=14, fontweight="bold", pad=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_preprocessing_pipeline.png", dpi=300,
            bbox_inches="tight")
plt.show()

---

# 12. Transform the Data

### What are we doing?

Now we fit the preprocessing pipeline on the training data and use it to transform both datasets.

The key distinction is between:

`fit_transform()`

and

`transform()`.

### Training data

`fit_transform()` does two things:

1. **Fit** — learn the preprocessing rules from the training data:
   - numeric medians
   - engineered-feature imputation values
   - category lists
   - scaling means and standard deviations
2. **Transform** — apply those rules to the training rows.

### Test data

For the test set we use only:

`transform()`

No new medians, category lists, or scaling statistics are calculated.

The test customers therefore receive exactly the rules learned from the training customers.

### Why is this important?

This is one of the most important ideas in the entire project:

> **The test set is transformed, not fitted.**

That keeps the test set independent and makes its later performance estimate meaningful.


In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

X_train_unscaled = preprocessor_unscaled.fit_transform(X_train)
X_test_unscaled = preprocessor_unscaled.transform(X_test)

print(f"X_train_processed: {X_train_processed.shape}")
print(f"X_test_processed:  {X_test_processed.shape}")
print(f"X_train_unscaled:  {X_train_unscaled.shape}")
print(f"X_test_unscaled:   {X_test_unscaled.shape}")

---

# 13. Verify the Final Dataset

### What are we checking?

Before saving anything, we verify that the transformed data has the properties required for modeling.

The checks confirm that:

- training and test data contain no missing values
- no infinite values remain
- identifier and KPI columns were removed
- the original engineered source columns were removed where intended
- training and test have exactly the same feature columns
- the final feature count matches the pipeline definition
- imputation values were learned from the training data

The feature count is taken directly from the transformed matrix rather than typed manually. This makes the verification more robust if the feature set changes later.

### Why use assertions?

An assertion turns an expectation into a test.

If a future change accidentally reintroduces a leakage column, changes the feature count, or breaks the train/test alignment, the notebook stops immediately instead of quietly producing a different dataset.

This section is therefore a small quality-control checkpoint before the artifacts are saved.


In [ ]:
# Counts are read from the transformed matrix, not typed in.
feature_names = list(X_train_processed.columns)
n_indicators = len(feature_names) - n_numeric_total

print(f"Raw predictor columns:    {X.shape[1]}")
print(f"  original numeric kept:  {n_numeric_kept}")
print(f"  engineered:             {n_engineered}")
print(f"  categorical indicators: {n_indicators}")
print(f"Final model features:     {len(feature_names)}")
print()
print("Numeric features:")
print("   " + ", ".join(feature_names[:n_numeric_total]))
print("Indicator features:")
print("   " + ", ".join(feature_names[n_numeric_total:]))

In [ ]:
# The medians stored by the fitted imputer should match the training medians.
fitted_imputer = preprocessor.named_transformers_["numeric"].named_steps["impute"]
learned_medians = pd.Series(fitted_imputer.statistics_, index=NUMERIC_FEATURES)
imputer_from_train = np.allclose(learned_medians[affected_columns],
                                 X_train[affected_columns].median())

leakage_columns = []
for name in feature_names:
    if name in KPI_COLUMNS + IDENTIFIER_COLUMNS + [TARGET]:
        leakage_columns.append(name)

remaining_sources = []
for column in SOURCE_COLUMNS_TO_DROP:
    if column in feature_names:
        remaining_sources.append(column)

# Exact duplicate records were removed before the train/test split. The check
# runs on the rows that survived that step, not on the raw file.
assert df[rows_to_keep].duplicated().sum() == 0, "Exact duplicate records remain in the modeling data."

checks = [
    ("No missing values in train", int(X_train_processed.isna().sum().sum()) == 0),
    ("No missing values in test", int(X_test_processed.isna().sum().sum()) == 0),
    ("No infinite values", bool(np.isfinite(X_train_processed.to_numpy()).all()
                                and np.isfinite(X_test_processed.to_numpy()).all())),
    ("No identifier or KPI columns", len(leakage_columns) == 0),
    ("No exact source columns retained", len(remaining_sources) == 0),
    ("Train and test columns identical",
     list(X_train_processed.columns) == list(X_test_processed.columns)),
    ("Feature count matches the pipeline definition",
     len(feature_names) == n_expected_features),
    ("Imputer values learned from training data only", bool(imputer_from_train)),
]

results = pd.DataFrame(checks, columns=["check", "passed"]).set_index("check")
print(results.to_string())

assert results.passed.all(), "One or more preprocessing checks failed."
print("\nAll checks passed.")

---

# 14. Save Preprocessing Artifacts

### What are we saving?

The notebook saves the objects Notebook 3 will need:

- `preprocessor.joblib`
- `preprocessor_unscaled.joblib`
- `feature_names.csv`
- the raw train/test splits
- both versions of the target

The business figures are **not** saved here. Measures such as funding gap and readiness belong after a prediction has been produced, because they depend on the predicted retirement fund.

### Why save the fitted pipeline?

The fitted pipeline contains the preprocessing rules learned from the training data.

Saving it means Notebook 3 does not need to rebuild those rules. More importantly, the same transformations can later be applied to new customers.

The feature-engineering step is stored in `src/feature_engineering.py`, so that reusable module must remain available when the saved pipeline is loaded.

The notebook also reloads the saved pipeline in a separate Python process. This checks that the artifact does not depend on temporary variables that exist only inside this notebook.


In [ ]:
joblib.dump(preprocessor, ARTIFACTS_DIR / "preprocessor.joblib")
joblib.dump(preprocessor_unscaled, ARTIFACTS_DIR / "preprocessor_unscaled.joblib")

pd.Series(feature_names, name="feature").to_csv(
    ARTIFACTS_DIR / "feature_names.csv", index=False)

# Raw splits, so Notebook 3 can rebuild any matrix through the pipeline.
X_train.to_csv(ARTIFACTS_DIR / "X_train.csv.gz")
X_test.to_csv(ARTIFACTS_DIR / "X_test.csv.gz")

pd.DataFrame({"log_target": y_train, "target_dollars": y_train_dollars}).to_csv(
    ARTIFACTS_DIR / "y_train.csv")
pd.DataFrame({"log_target": y_test, "target_dollars": y_test_dollars}).to_csv(
    ARTIFACTS_DIR / "y_test.csv")

for artifact_path in sorted(ARTIFACTS_DIR.iterdir()):
    size_mb = artifact_path.stat().st_size / 1024**2
    print(f"   {artifact_path.name:<30} {size_mb:>6.2f} MB")

In [ ]:
# Load the pipeline in a separate process and re-transform the test set there.
reload_script = """
import joblib
import pandas as pd

preprocessor = joblib.load("artifacts/preprocessor.joblib")
X_test = pd.read_csv("artifacts/X_test.csv.gz", index_col=0)
transformed = preprocessor.transform(X_test)

print(transformed.shape[0])
print(transformed.shape[1])
print(",".join(transformed.columns))
"""

result = subprocess.run(
    [sys.executable, "-c", reload_script],
    cwd=PROJECT_ROOT, capture_output=True, text=True,
)

if result.returncode != 0:
    raise RuntimeError(f"Reload failed:\n{result.stderr}")

reloaded_rows, reloaded_columns, reloaded_names = result.stdout.strip().split("\n")

print(f"Rows reproduced:          {int(reloaded_rows) == X_test_processed.shape[0]}")
print(f"Columns reproduced:       {int(reloaded_columns) == X_test_processed.shape[1]}")
print(f"Feature names reproduced: {reloaded_names.split(',') == feature_names}")

assert int(reloaded_rows) == X_test_processed.shape[0]
assert int(reloaded_columns) == X_test_processed.shape[1]
assert reloaded_names.split(",") == feature_names
print("\nThe saved pipeline reproduces the test matrix in a separate process.")

---

# 15. Summary

This notebook converts the raw customer records into a modeling-ready dataset while keeping the test set protected from learned preprocessing decisions.

### What changed?

**Removed**

- `CustomerID`
- `Funding_Gap`
- `Readiness_Score`
- `RetirementReady`
- 150 exact duplicate records
- `DesiredRetirementAge`, `YearsExperience`, and `ExpectedInflation` after their information was represented in engineered features

**Engineered**

- `YearsUntilRetirement`
- `CareerStartAge`
- `SalaryBasedContribution`
- `RealExpectedReturn`
- `DebtToIncomeRatio`

Customers with zero salary receive a missing `DebtToIncomeRatio`, which is then handled by imputation. Their unemployed status remains available through categorical encoding.

**Handled missing values**

- Numeric values → training median
- Categorical values → most frequent training category

**Encoded categorical variables**

Categories were converted to binary indicators, with one level dropped from each categorical feature to avoid perfect redundancy. Unknown categories are ignored rather than causing the pipeline to fail.

**Created two numeric representations**

- Scaled → suitable for linear and distance-based models
- Unscaled → suitable for tree-based models

### Most important rule

The train/test split happens before preprocessing is fitted.

The pipeline learns medians, category lists, and scaling statistics from training data only. The test set is only transformed with those learned rules.

### Artifacts for Notebook 3

The following are now ready:

- `preprocessor.joblib`
- `preprocessor_unscaled.joblib`
- `feature_names.csv`
- raw train/test splits
- both target versions

The data is now ready for **baseline model training and comparison in Notebook 3**.
